# Minimal Gravimetry ML: methodology

This notebook is the reference for the project. The task is to reconstruct binary shapes from complex boundary-gradient measurements. The production evaluation keeps validation and test measurements clean while adding independent absolute Gaussian noise to training gradients. A separate robustness section evaluates a fixed trained model on noisy test measurements without using those results for model selection.

## System workflow

The three model paradigms are:

- One-stage: gradient features -> mask logits.
- Two-stage: gradient features -> coefficients -> mask logits.
- Three-stage experimental system: gradient features -> coefficients -> general or specialist mask.

The three-stage specialist route uses shape labels in the current implementation and is therefore documented as experimental rather than deployment-valid.

## Measurement and noise model

For each complex gradient measurement `g`, the training-only noise convention is:

`g_noisy = g_clean + e_real + i e_imag`, where `e_real` and `e_imag` are independent `Normal(0, sigma^2)` variables. `sigma` is an absolute standard deviation and is not scaled by signal magnitude.

Official metrics use clean validation and test gradients. Supplementary robustness curves use the exact test noise levels `0.0`, `0.001`, `0.0025`, `0.005`, and `0.01`.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import OneModelRunConfig
from datasets import build_two_stage_datasets
from experiments import feature_matrix_with_noise

OUTPUT = ROOT / 'output' / 'methodology'
OUTPUT.mkdir(parents=True, exist_ok=True)
config = OneModelRunConfig(N=10, training_samples=500, validation_samples=100, test_samples=100, noise_sigma=0.001, seed=42)
bundle = build_two_stage_datasets(config)
print({name: split.gradient_data.shape for name, split in bundle.splits().items()})

In [ ]:
clean = bundle.train.gradient_data[0]
noisy = feature_matrix_with_noise(bundle.train.gradient_data[:1], 0.01, seed=42)[0]
half = clean.size // 2
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(clean[:half], label='clean real')
axes[0].plot(noisy[:half], label='noisy real', alpha=0.8)
axes[0].set_title('Real gradient features')
axes[0].set_xlabel('Measurement index')
axes[0].legend()
axes[1].imshow(bundle.train.masks[0].reshape(config.grid_size, config.grid_size), cmap='gray')
axes[1].set_title('Training target mask')
axes[1].axis('off')
figure.tight_layout()
figure.savefig(OUTPUT / 'clean_noisy_example.png', dpi=180)
plt.show()

## Dataset splitting and leakage prevention

The generator samples training, validation, and random-test shapes sequentially from one seeded random generator. Training shape weights and noise replication apply only to the training split. Validation and random-test splits use clean gradients and are never used for optimization. The fixed benchmark contains deterministic named shapes and is reported separately because it is a diagnostic set rather than a random estimate.

Threshold selection uses validation IoU. The random-test labels are read only after training and threshold selection. No noisy-test result is used to select a model.

## Metrics and reporting

The primary metric is mean per-example mask IoU. Per-shape IoU is reported to expose bottlenecks such as annuli, rectangles, or disconnected circles. BCE and Dice are training objectives; true-coefficient two-stage results are diagnostics only. Every active notebook writes configuration, checkpoints, histories, CSV/JSON metrics, plots, and qualitative outputs under `output/`.